<a href="https://colab.research.google.com/github/edwardoughton/satellite-image-analysis/blob/main/06_01_ggs416_26_03_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛰️GGS416 Week 6: Object-Based Image Classification 🛰️

This week we transition from last week's *pixel-based classification to object-based image analysis (OBIA).

The main methodological shift is not a brand-new classifier. Instead, the key change is that we stop treating every pixel as an independent observation and begin grouping nearby pixels into meaningful image objects before classification.

Lesson plan:
1. Scene setup and visual audit (~20 min)
2. Spectral feature stack and weak labels (~20 min)
3. Quick pixel baseline and its limitations (~20 min)
4. SLIC segmentation and parameter tuning (~20 min)
5. Object features, object classification, and comparison (~20 min)
6. Syntax test (~30 min)


## Learning objectives

By the end of class, students should be able to:

1. Explain why pixel level classification often breaks down in high resolution imagery.
2. Access, clip and describe NAIP imagery, including why this data is great for separating water, vegetation, built surfaces, etc.
3. Generate SLIC image segments and explain what the main parameters control.
4. Extract object summary features from segments.
5. Explain how a Random Forest model behaves when the training unit changes from pixels to objects.
6. Compare a pixel baseline against an object classification map and diagnose likely failure modes.

## Install packages

In [ ]:
#!pip -q install pystac-client planetary-computer requests rasterio scikit-learn scikit-image matplotlib

## Import packages and setup helper functions

We intentionally reuse the same Planetary Computer and raster IO pattern from the earlier notebooks so that the class can focus on the new concept, e.g., objects as units of analysis.

The helper functions below query and download a NAIP scene, stretch RGB values for display, visualize intermediate image products clearly, and convert class labels into simple RGB maps.

The download helper is for a single image (later in the class we might progress to mosaicing).

In [ ]:
# Import packages
import os
import numpy as np
import matplotlib.pyplot as plt

import requests
from pystac_client import Client
import planetary_computer as pc

import rasterio
from rasterio.merge import merge
from rasterio.windows import from_bounds
from rasterio.warp import transform_bounds

# Get necessary scikit-learn functions
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

# Get necessary scikit-image functions
from skimage.segmentation import slic, mark_boundaries

np.random.seed(42) # Set seed in case we need repeatable random results

In [ ]:
# These are a set of repeatable functions to save space.
def stretch_rgb(rgb_bands):
    """Percentile contrast stretch on a 3-band RGB raster array
    and scales the result to the range 0–1 for visualization."""
    rgb_hwc = np.moveaxis(rgb_bands, 0, -1).astype("float32")
    p2, p98 = np.nanpercentile(rgb_hwc, (2, 98))
    return np.clip((rgb_hwc - p2) / (p98 - p2 + 1e-6), 0, 1)


def show_image(img, title, cmap=None, figsize=(6, 6)):
    """Label and show image using matplotlib"""
    plt.figure(figsize=figsize)
    plt.imshow(img, cmap=cmap)
    plt.title(title)
    plt.axis("off")
    plt.show()


def label_to_rgb(label_img, color_map, nodata=None):
    """Converts a classified (label) raster into a 3-band RGB
    image using a lookup color map"""
    out = np.zeros((*label_img.shape, 3), dtype=np.float32)
    for class_id, color in color_map.items():
        out[label_img == class_id] = color
    if nodata is not None:
        out[label_img == nodata] = (0.0, 0.0, 0.0)
    return out

def majority_label(values, ignore_value=255):
    """Computes the majority class (mode) from an array of label values,
    optionally ignoring a specified value (typically a NoData class)."""
    values = values[values != ignore_value]
    if values.size == 0:
        return ignore_value
    ids, counts = np.unique(values, return_counts=True)
    return int(ids[np.argmax(counts)])

## Download a NAIP scene

We narrow the search in two ways:

- Spatially, using a bounding box centered on our chosen location.
- Temporally, using a time window of our choosing.

Let us set up `my_download_function` to allow us to access specific NAIP imagery.

In [ ]:
# Specify our download function
# Pretty much the same as our previous class code for obtaining NAIP data
def my_download_function(bbox, datetime, directory):

    os.makedirs(directory, exist_ok=True) # Create directory if it doesn't exist

    # Get our catalog
    catalog = Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )

    # Now search based on our imagery type, bbox, date etc.
    items = list(
        catalog.search(
            collections=["naip"],
            bbox=bbox,
            datetime=datetime,
            limit=12,
            method="POST",
        ).items()
    )

    if not items:
        raise ValueError("No NAIP items found for the requested bbox/dates.")

    # Let us now get a single item from those returned
    latest_item = max(items, key=lambda item: item.datetime)
    target_date = latest_item.datetime.date()
    selected_item = [item for item in items if item.datetime and
                     item.datetime.date() == target_date][0]
    print("Acquiring:         ", selected_item)

    # Get the href, and specify out path
    href = pc.sign(selected_item).assets["image"].href
    filename = os.path.basename(href.split("?")[0])
    out_path = os.path.join(directory, filename)

    # Finally, we check if we have downloaded it yet, and if not, execute
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        print("Already downloaded:", out_path)
    else:
        with requests.get(href, stream=True) as r:
            r.raise_for_status()
            with open(out_path, "wb") as f:
                for chunk in r.iter_content(1024 * 1024):
                    if chunk:
                        f.write(chunk)
        print("Saved:", out_path)

    return out_path

# minlon, minlat, maxlon, maxlat
bbox = (-77.104146, 38.90558, -77.091331, 38.911665)
datetime = "2023-05-01/2023-09-30"
directory = "naip_downloads"

naip_path = my_download_function(bbox, datetime, directory)
print("Using file:", naip_path)

## Crop the full tile and inspect the scene

NAIP is well suited to object based work because it has high enough spatial resolution for building edges, roads, tree masses, and river boundaries to be visually meaningful.

However, the full tile is too large for a live class, so we crop to a smaller central area.

We also want you to be able to quickly iterate this code, changing parameters as desired, so you can inspect the impacts on the image outputs. This crop reduces runtime, making segmentation faster.

In [ ]:
# Crop the image we just downloaded

# Make the new directory if it does not already exist
os.makedirs("naip_cropped", exist_ok=True)

# Specify out path to write the clipped image to
out_path = os.path.join("naip_cropped", f"crop_{os.path.basename(naip_path)}")

# Open our image with rasterio
with rasterio.open(naip_path) as src:

    print("CRS:", src.crs) # Check the current CRS

    # Use the bbox so the crop stays central to our Area Of Interest (AOI)
    # Better than a center-crop (AOI may not be in the middle of the image)
    minlon, minlat, maxlon, maxlat = bbox

    # Convert to lat/lon coordinates, e.g., from "EPSG:26918" to "EPSG:4326"
    minx, miny, maxx, maxy = transform_bounds(
        "EPSG:4326",
        src.crs,
        minlon,
        minlat,
        maxlon,
        maxlat,
    )

    # Get our window of interest shape
    win = from_bounds(minx, miny, maxx, maxy, src.transform)

    # Crop to our window of interest
    img = src.read(window=win).astype("float32")

    # Copy metadata
    profile = src.profile
    profile.update(
        height=img.shape[1],
        width=img.shape[2],
        transform=rasterio.windows.transform(win, src.transform),
    )

# Write out the cropped image to our desired path
with rasterio.open(out_path, "w", **profile) as dst:
    dst.write(img)

print("Wrote:", out_path)

rgb01 = stretch_rgb(img[:3])
show_image(rgb01, "Summer NAIP RGB: Arlington (cropped)", figsize=(7, 7))

# Handle IR band, if present
if img.shape[0] > 3:
    nir = img[3]
    nir01 = np.clip(nir / (np.nanpercentile(nir, 98) + 1e-6), 0, 1)
    show_image(nir01, "NIR band (contrast stretch)", cmap="gray", figsize=(7, 7))

### Exercise: Visual audit

Inspect the maps and identify:

1. The clearest river area.
2. The densest tree canopy.
3. The main building clusters.
4. At least one other object in the river area.
5. Two places where pixel classification is likely to be unstable because of mixed shadows, edges, etc.

Consider spectrally uniform areas (open water versus dense tree canopy, versus buildings, roads, etc.).

## Build a spectral feature stack and create  labels

Before we move into objects, we first rebuild a familiar spectral feature space. This gives us a bridge from last week and creates a simple set of labels for use.

The feature stack combines visible bands, NIR, NDVI, NDWI, and brightness. NDVI is especially useful in summer because tree canopy is usually more distinct. NDWI helps separate the Potomac River, and brightness often helps isolate bright built surfaces such as roads, roofs, and bridge decks.

The (pseudo) labels we produce are not necessarily ground truth, but are a fast classroom approximation so that we can compare two workflows consistently, e.g., classify each pixel directly, and classify objects after segmentation.

In [ ]:
# Develop our feature stack

# Get rgb layers
r = img[0]
g = img[1]
b = img[2]

# Get NIR if present
nir = img[3] if img.shape[0] > 3 else None
if nir is None:
    raise ValueError("This NAIP tile does not include NIR band, but NDVI/NDWI need it.")

# Define spectral indices
ndvi = (nir - r) / (nir + r + 1e-6)
ndwi = (g - nir) / (g + nir + 1e-6)
brightness = (r + g + b) / 3.0

# Specify our feature stack and feature names
feature_stack = np.dstack([r, g, b, nir, ndvi, ndwi, brightness])
feature_names = ["R", "G", "B", "NIR", "NDVI", "NDWI", "Brightness"]
print(feature_stack)


In [ ]:
# Visualize our features (e.g., spectral indices)
fig, ax = plt.subplots(2, 2, figsize=(12, 10))
ax[0, 0].imshow(rgb01)
ax[0, 0].set_title("RGB")
ax[0, 0].axis("off")

im1 = ax[0, 1].imshow(ndvi, cmap="RdYlGn", vmin=-1, vmax=1)
ax[0, 1].set_title("NDVI")
ax[0, 1].axis("off")
plt.colorbar(im1, ax=ax[0, 1], fraction=0.046)

im2 = ax[1, 0].imshow(ndwi, cmap="BrBG", vmin=-1, vmax=1)
ax[1, 0].set_title("NDWI")
ax[1, 0].axis("off")
plt.colorbar(im2, ax=ax[1, 0], fraction=0.046)

im3 = ax[1, 1].imshow(brightness, cmap="gray")
ax[1, 1].set_title("Brightness")
ax[1, 1].axis("off")
plt.colorbar(im3, ax=ax[1, 1], fraction=0.046)

plt.tight_layout()
plt.show()


In [ ]:
# Now define our labels and plot
labels = np.full(ndvi.shape, 255, dtype=np.uint8)

# Water: Strong positive NDWI is a quick way to isolate water areas.
water_mask = ndwi > 0.10

# Vegetation: Summer NDVI is usually strong, so tree canopy should be clear.
veg_mask = (ndvi > 0.25) & (~water_mask)

# Built-up area: Brighter surfaces often signal road/rooftop reflections.
built_mask = (brightness > np.percentile(brightness, 70)) & (ndvi < 0.20) & (~water_mask)

# Set labels as numbers
labels[water_mask] = 0
labels[veg_mask] = 1
labels[built_mask] = 2

# Set class names and colors
class_names = {0: "water", 1: "vegetation", 2: "built"}
class_colors = {
    0: (0.10, 0.35, 0.90),
    1: (0.10, 0.70, 0.20),
    2: (0.78, 0.48, 0.25),
}

label_vis = label_to_rgb(labels, class_colors, nodata=255)

# Now plot to show original, our labels, and labels overlaid
fig, ax = plt.subplots(1, 3, figsize=(16, 6))
ax[0].imshow(rgb01)
ax[0].set_title("RGB")
ax[0].axis("off")

ax[1].imshow(label_vis)
ax[1].set_title("Estimated labels")
ax[1].axis("off")

ax[2].imshow(rgb01)
ax[2].imshow(label_vis, alpha=0.45)
ax[2].set_title("Label overlay")
ax[2].axis("off")

plt.tight_layout()
plt.show()

print("Feature stack shape:", feature_stack.shape)
print("Labeled pixels:", int((labels != 255).sum()))

### Exercise: Feature reasoning

Using the RGB, NDVI, NDWI, and brightness maps estimate:

1. Which parts of the scene should be easiest to classify as water?
2. Which parts of the scene should be easiest to classify as vegetation?
3. Which built features may still be confusing because they share brightness with artificial materials (concrete, rooftops, etc.)?
4. Which zones will be hardest because of shadow edges or mixed urban materials?

## 5) Build a quick pixel baseline with Random Forest

Before we move to objects, we build a short pixel baseline. This is important  because it gives us a direct comparison point.

If we change both the classifier and the unit of analysis at the same time, you cannot tell which change caused the output to improve or degrade.

So we deliberately keep the same model family from last week (Random Forest classifier). Hence, the important methodological change this week is not a new algorithm, but the unit of analysis.

We know already that a Random Forest is an ensemble of decision trees. Each tree sees a bootstrap sample of the training data, each split only considers a subset of features, and the final prediction is produced by majority vote across trees. It handles nonlinear relationships, is fairly robust to noisy labels, tolerates mixed feature scales, and trains fast enough for a live class.

It still struggles in a pixel workflow because pixel-wise predictions ignore spatial contiguity. In high-resolution urban imagery, shadows and within-object variability often produce poor maps even when the classifier itself is behaving as designed.

In [ ]:
# Train and test our random forest (covered last week)
rng = np.random.default_rng(42)

X_all = feature_stack.reshape(-1, feature_stack.shape[-1])
y_all = labels.reshape(-1)

valid_idx = np.where(y_all != 255)[0]
X_valid = X_all[valid_idx]
y_valid = y_all[valid_idx]

# Balanced sampling keeps the classroom example stable and avoids one class dominating the demo.
samples_per_class = 2500
chosen = []
for cid in sorted(class_names):
    idx = np.where(y_valid == cid)[0]
    take = min(samples_per_class, len(idx))
    chosen.append(rng.choice(idx, size=take, replace=False))

chosen = np.concatenate(chosen)
X = X_valid[chosen]
y = y_valid[chosen]

# The train/test split stays simple on purpose so the modeling story is easy to follow.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# This is intentionally a straightforward baseline model rather than a heavily tuned one.
pixel_rf = RandomForestClassifier(
    n_estimators=10,
    random_state=42,
    n_jobs=-1,
)
pixel_rf.fit(X_train, y_train)

pixel_test_pred = pixel_rf.predict(X_test)
print(f"Pixel model accuracy: {accuracy_score(y_test, pixel_test_pred):.3f}")
print()
print(classification_report(y_test, pixel_test_pred, target_names=[class_names[i] for i in sorted(class_names)]))


In [ ]:
# Explore our confusion matrix
cm = confusion_matrix(y_test, pixel_test_pred, labels=sorted(class_names))
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    cm,
    display_labels=[class_names[i] for i in sorted(class_names)],
).plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("Pixel Baseline Confusion Matrix")
plt.tight_layout()
plt.show()

# Now try predict all pixels
pixel_pred_full = pixel_rf.predict(X_all).reshape(labels.shape)
pixel_vis = label_to_rgb(pixel_pred_full, class_colors)



In [ ]:
# Visualize results
fig, ax = plt.subplots(1, 3, figsize=(16, 6))
ax[0].imshow(rgb01)
ax[0].set_title("RGB")
ax[0].axis("off")

ax[1].imshow(pixel_vis)
ax[1].set_title("Pixel baseline")
ax[1].axis("off")

ax[2].imshow(rgb01)
ax[2].imshow(pixel_vis, alpha=0.45)
ax[2].set_title("Pixel overlay")
ax[2].axis("off")

plt.tight_layout()
plt.show()

## Segment the image into superpixels with SLIC

Now we switch from classifying pixels directly to first creating image objects.

Here we use Simple Linear Iterative Clustering (SLIC), which is a classic superpixel method (covered previously).

Remember, SLIC is a segmentation algorithm, not a classifier. It does not know what is water, vegetation, built-up area, etc. Instead, it groups nearby pixels that are similar in color and close in space.

You can explain SLIC in practical terms. First, SLIC begins from an approximately regular grid of seed locations, then iteratively adjusts those groupings using both spectral similarity and spatial proximity. The result is a set of compact image regions, often called "superpixels".

The main model parameters are `n_segments` (approximate number of output regions), `compactness` (how strongly the algorithm prefers regular spatial shape versus color similarity), and `sigma` (optional smoothing before segmentation). In this scene, students should look carefully at river boundaries, tree masses, building edges, and roads.

In [ ]:
# Specify our SLIC superpixels
slic_labels = slic(
    rgb01,
    n_segments=350,
    compactness=12.0,
    sigma=1.0,
    start_label=1,
)

slic_overlay = mark_boundaries(rgb01, slic_labels, color=(1, 1, 0))

labels_low = slic(rgb01, n_segments=350, compactness=5.0, sigma=1.0, start_label=1)
labels_high = slic(rgb01, n_segments=350, compactness=25.0, sigma=1.0, start_label=1)

overlay_low = mark_boundaries(rgb01, labels_low, color=(1, 1, 0))
overlay_high = mark_boundaries(rgb01, labels_high, color=(1, 1, 0))

segment_mean_rgb = np.zeros_like(rgb01)
for seg_id in np.unique(slic_labels):
    mask = slic_labels == seg_id
    segment_mean_rgb[mask] = rgb01[mask].mean(axis=0)

fig, ax = plt.subplots(2, 2, figsize=(14, 12))
ax[0, 0].imshow(rgb01)
ax[0, 0].set_title("RGB")
ax[0, 0].axis("off")

ax[0, 1].imshow(slic_overlay)
ax[0, 1].set_title("SLIC boundaries")
ax[0, 1].axis("off")

ax[1, 0].imshow(overlay_low)
ax[1, 0].set_title("Lower compactness")
ax[1, 0].axis("off")

ax[1, 1].imshow(overlay_high)
ax[1, 1].set_title("Higher compactness")
ax[1, 1].axis("off")

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(13, 6))
ax[0].imshow(slic_labels, cmap="tab20")
ax[0].set_title("SLIC label map")
ax[0].axis("off")

ax[1].imshow(segment_mean_rgb)
ax[1].set_title("Segment mean-color reconstruction")
ax[1].axis("off")

plt.tight_layout()
plt.show()

print("Number of SLIC segments:", len(np.unique(slic_labels)))

### Exercise: Tune SLIC by inspection

Rerun the segmentation cell after changing one parameter at a time.

Suggested trials:

1. `n_segments = 150`
2. `n_segments = 600`
3. `compactness = 3`
4. `compactness = 30`

Which setting best captures the objects in the image? E.g., which one best follows the river edges, and which one handles road corridors most cleanly, etc.

## Convert segments into object observations

This is the core OBIA step. Once the segmentation is complete, each segment becomes a single observation in a new training table.

Conceptually, we are now summarizing the pixel values inside each segment, treating the segment as an object with its own attributes, and **assigning that object a label based on the majority class of the pixel labels inside it**.

Mean features reduce some pixel noise. The object boundary then imposes spatial coherence and each area becomes a meaningful object feature.

However, if a segment crosses a shoreline, shadow edge, or mixed land use boundary, the majority-vote label can be misleading.

In [ ]:
# Create our feature table
unique_segments = np.unique(slic_labels)

segment_ids = []
segment_features = []
segment_targets = []
segment_ndvi_mean = np.zeros_like(ndvi, dtype=np.float32)

for seg_id in unique_segments:
    mask = slic_labels == seg_id

    # Each segment becomes one row in a new object-level feature table.
    feat = [
        float(r[mask].mean()),
        float(g[mask].mean()),
        float(b[mask].mean()),
        float(nir[mask].mean()),
        float(ndvi[mask].mean()),
        float(ndwi[mask].mean()),
        float(brightness[mask].mean()),
        float(mask.sum()),
    ]

    segment_ids.append(seg_id)
    segment_features.append(feat)
    segment_targets.append(majority_label(labels[mask], ignore_value=255))
    segment_ndvi_mean[mask] = feat[4]

segment_ids = np.asarray(segment_ids)
segment_features = np.asarray(segment_features, dtype=np.float32)
segment_targets = np.asarray(segment_targets)
segment_feature_names = [
    "mean_R",
    "mean_G",
    "mean_B",
    "mean_NIR",
    "mean_NDVI",
    "mean_NDWI",
    "mean_Brightness",
    "area",
]

segment_label_image = np.full_like(labels, 255)
for seg_id, target in zip(segment_ids, segment_targets):
    if target != 255:
        segment_label_image[slic_labels == seg_id] = target

segment_label_vis = label_to_rgb(segment_label_image, class_colors, nodata=255)
print(len(segment_label_vis))

In [ ]:
# Visualize our image segments
fig, ax = plt.subplots(1, 3, figsize=(17, 6))
im = ax[0].imshow(segment_ndvi_mean, cmap="RdYlGn", vmin=-1, vmax=1)
ax[0].set_title("Mean NDVI per segment")
ax[0].axis("off")
plt.colorbar(im, ax=ax[0], fraction=0.046)

ax[1].imshow(segment_label_vis)
ax[1].set_title("Majority label per segment")
ax[1].axis("off")

ax[2].imshow(rgb01)
ax[2].imshow(segment_label_vis, alpha=0.45)
ax[2].set_title("Segment label overlay")
ax[2].axis("off")

plt.tight_layout()
plt.show()

valid_segments = segment_targets != 255
print("Object feature matrix shape:", segment_features.shape)
print("Segments with labels:", int(valid_segments.sum()))
print("Segments without labels:", int((~valid_segments).sum()))

### Exercise: Inspect segment label quality

Examine the segment label overlay and identify where majority vote labeling worked well, and equally, where it performed poorly.


## Train an object based Random Forest and compare it to the pixel map

We now train another Random Forest classifier. However, while the algorithm is the same, the training rows are different.

We now use features aggregated per object instead of per pixel.

That is the key contrast this week.

The object model should produce smoother and more spatially coherent class regions, but it is also sensitive to segmentation quality.

If segmentation is too coarse, several real classes may be merged into one segment. If segmentation is too fine, the workflow starts to resemble pixel classification again.

The classifier also inherits any errors from segmentation and from majority vote segment labeling.

In [ ]:
X_seg = segment_features[valid_segments]
y_seg = segment_targets[valid_segments]

X_train_seg, X_test_seg, y_train_seg, y_test_seg = train_test_split(
    X_seg, y_seg, test_size=0.3, random_state=42, stratify=y_seg
)

object_rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
)
object_rf.fit(X_train_seg, y_train_seg)

segment_test_pred = object_rf.predict(X_test_seg)
print(f"Object model accuracy: {accuracy_score(y_test_seg, segment_test_pred):.3f}")
print()
print(classification_report(y_test_seg, segment_test_pred, target_names=[class_names[i] for i in sorted(class_names)]))


In [ ]:
# View our confusion matrix
cm = confusion_matrix(y_test_seg, segment_test_pred, labels=sorted(class_names))
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    cm,
    display_labels=[class_names[i] for i in sorted(class_names)],
).plot(ax=ax, cmap="Greens", colorbar=False)
plt.title("Object Based Confusion Matrix")
plt.tight_layout()
plt.show()

segment_predictions = object_rf.predict(segment_features)
object_pred_map = np.zeros_like(labels)
for seg_id, pred in zip(segment_ids, segment_predictions):
    object_pred_map[slic_labels == seg_id] = pred

object_vis = label_to_rgb(object_pred_map, class_colors)
object_overlay = mark_boundaries(object_vis, slic_labels, color=(1, 1, 1))



In [ ]:
# Visualize our results
fig, ax = plt.subplots(2, 2, figsize=(14, 12))
ax[0, 0].imshow(rgb01)
ax[0, 0].set_title("RGB")
ax[0, 0].axis("off")

ax[0, 1].imshow(pixel_vis)
ax[0, 1].set_title("Pixel baseline")
ax[0, 1].axis("off")

ax[1, 0].imshow(object_vis)
ax[1, 0].set_title("Object-based classification")
ax[1, 0].axis("off")

ax[1, 1].imshow(object_overlay)
ax[1, 1].set_title("Object map with segment boundaries")
ax[1, 1].axis("off")

plt.tight_layout()
plt.show()

pixel_counts = {class_names[cid]: int((pixel_pred_full == cid).sum()) for cid in sorted(class_names)}
object_counts = {class_names[cid]: int((object_pred_map == cid).sum()) for cid in sorted(class_names)}
print("Pixel class counts:", pixel_counts)
print("Object class counts:", object_counts)

### Exercise: Compare the final maps

Compare the pixel and object outputs and answer:

1. Where does the object based map visibly reduce "salt-and-pepper" noise?
2. Where does segmentation introduce new errors?
3. Which result looks more trustworthy along the river edge?
4. Which result handles buildings and shadows better?
5. Would you expect the same object based advantage on lower-resolution imagery such as Landsat?